In [3]:
!pip install protobuf sentencepiece

In [5]:
%%writefile "C:/Users/nivsa/Generation of Synthetic Training Data/embedded/extraction_engine/aligner.py"

import os
import glob
import json
import re
import time
from typing import List, Dict, Tuple, Any, Optional
from bs4 import BeautifulSoup, NavigableString, Tag
from dataclasses import dataclass
import html


GENERIC_LABELS = {"PARAMETER", "VALUE", "MIN", "MAX", "TYP", "UNIT", "CONDITION"}

RELATION_TYPES = {
    "has_value": "has_value",
    "has_unit": "has_unit",
    "has_min": "has_min",
    "has_max": "has_max",
    "has_typ": "has_typ",
    "has_condition": "has_condition",
}


@dataclass
class Span:
    start: int
    end: int
    label: str
    text: str


@dataclass
class Token:
    text: str
    start: int
    end: int
    span_label: str = "O"


class PreprocessingEngine:

    def __init__(self, tokenizer=None):
        self.spans: List[Span] = []
        self.clean_text: str = ""
        self.tokens: List[Token] = []
        self.custom_tokenizer = tokenizer

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def process(self, html_content: str, jsonl_relations: List[Dict] = None) -> List[Dict]:
        html_content = html.unescape(html_content)

        self.clean_text, self.spans = self._extract_text_and_spans(html_content)
        self.tokens = self._tokenize(self.clean_text)
        self._assign_bio_tags()
        samples = self._sliding_window(max_tokens=512, overlap=50)

        if jsonl_relations:
            samples = self._add_relations(samples, jsonl_relations)

        return samples

    # ------------------------------------------------------------------
    # DOM → clean text + spans
    # ------------------------------------------------------------------

    def _extract_text_and_spans(self, html_content: str) -> Tuple[str, List[Span]]:
        soup = BeautifulSoup(html_content, 'html.parser')

        segments: List[Tuple[str, Optional[str]]] = []
        self._walk_dom(soup, segments)

        raw_text_parts: List[str] = []
        pending_spans: List[Tuple[int, int, str]] = []
        cursor = 0

        # FIX #1 (artifact-shift bug): clean each fragment BEFORE recording its
        # offset. Cleaning after joining would shrink the string and push every
        # span index recorded below to the wrong position in the final text.
        for text_frag, label in segments:
            text_frag = self._clean_artifacts(text_frag)   # safe: per-fragment, no cross-span drift
            if not text_frag:
                continue
            raw_start = cursor
            raw_text_parts.append(text_frag)
            cursor += len(text_frag)
            if label is not None:
                pending_spans.append((raw_start, cursor, label))

        raw_text = "".join(raw_text_parts)

        clean_text, offset_map = self._normalize_whitespace_with_map(raw_text)

        spans: List[Span] = []
        for raw_start, raw_end, label in pending_spans:
            norm_start = offset_map[min(raw_start, len(offset_map) - 1)]
            last_char_idx = min(raw_end - 1, len(offset_map) - 1)
            norm_end = offset_map[last_char_idx] + 1

            surface = clean_text[norm_start:norm_end]
            if surface.strip():
                spans.append(Span(
                    start=norm_start,
                    end=norm_end,
                    label=self._normalize_label(label),
                    text=surface,
                ))

        return clean_text, spans

    def _walk_dom(self, node, segments: List[Tuple[str, Optional[str]]], in_table: bool = False):
        if isinstance(node, NavigableString):
            text = str(node)
            if text:
                segments.append((text, None))
            return

        tag_name = node.name if hasattr(node, 'name') else None

        # FIX #1: skip non-semantic tags — CSS/JS pollute the token stream
        if tag_name in ('style', 'script', 'head'):
            return

        # Table structure: inject \t between cells, \n after each row
        if tag_name == 'tr':
            first_cell = True
            for child in node.children:
                if hasattr(child, 'name') and child.name in ('td', 'th'):
                    if not first_cell:
                        segments.append(('\t', None))
                    first_cell = False
                    self._walk_dom(child, segments, in_table=True)
                else:
                    self._walk_dom(child, segments, in_table=True)
            segments.append(('\n', None))
            return

        if tag_name in ('td', 'th'):
            for child in node.children:
                self._walk_dom(child, segments, in_table=True)
            return

        # Labeled span — emit as a single tagged segment
        if tag_name == 'span' and node.get('data-label'):
            label = node.get('data-label')
            sub_segments: List[Tuple[str, Optional[str]]] = []
            for child in node.children:
                self._walk_dom(child, sub_segments, in_table)
            span_text = "".join(s for s, _ in sub_segments)
            if span_text.strip():
                segments.append((span_text, label))
            return

        # Block-level tags: inject spaces to prevent word concatenation
        BLOCK_TAGS = {'p', 'div', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6',
                      'li', 'br', 'table', 'thead', 'tbody', 'tfoot'}
        is_block = tag_name in BLOCK_TAGS if tag_name else False

        if is_block:
            segments.append((' ', None))
        for child in node.children:
            self._walk_dom(child, segments, in_table)
        if is_block and tag_name != 'br':
            segments.append((' ', None))

    # ------------------------------------------------------------------
    # FIX #4: artifact cleaning
    # ------------------------------------------------------------------

    @staticmethod
    def _clean_artifacts(text: str) -> str:
        # Remove Python list notation around values: ['-65 to 150'] → -65 to 150
        text = re.sub(r"\[(['\"])(.*?)\1\]", r'\2', text)

        # Deduplicate immediately repeated unit strings (e.g. µVrmsµVrms → µVrms)
        # Matches any non-whitespace run of 2-20 chars repeated back-to-back
        text = re.sub(r'(\b\S{2,20})\1\b', r'\1', text)

        return text

    # ------------------------------------------------------------------
    # Whitespace normalization with offset map
    # ------------------------------------------------------------------

    def _normalize_whitespace_with_map(self, raw: str) -> Tuple[str, List[int]]:
        """Collapse space runs to one; preserve \\t and \\n (table structure)."""
        norm_chars: List[str] = []
        offset_map: List[int] = []
        norm_idx = 0
        i = 0

        while i < len(raw):
            ch = raw[i]
            if ch == ' ':
                offset_map.append(norm_idx)
                if not norm_chars or norm_chars[-1] in (' ', '\t', '\n'):
                    pass  # collapse
                else:
                    norm_chars.append(' ')
                    norm_idx += 1
            else:
                norm_chars.append(ch)
                offset_map.append(norm_idx)
                norm_idx += 1
            i += 1

        result = "".join(norm_chars).strip()
        leading_stripped = len("".join(norm_chars)) - len("".join(norm_chars).lstrip())
        if leading_stripped:
            offset_map = [max(0, v - leading_stripped) for v in offset_map]

        return result, offset_map

    # ------------------------------------------------------------------
    # Label normalization
    # ------------------------------------------------------------------

    def _normalize_label(self, data_label: str) -> str:
        label = data_label.upper().strip()

        if label in GENERIC_LABELS:
            return label

        for suffix, canon in [('_MIN', 'MIN'), ('_MINIMUM', 'MIN'),
                               ('_MAX', 'MAX'), ('_MAXIMUM', 'MAX'),
                               ('_TYP', 'TYP'), ('_TYPICAL', 'TYP'),
                               ('_VALUE', 'VALUE'), ('_VAL', 'VALUE')]:
            if label.endswith(suffix):
                return canon

        if 'UNIT' in label:
            return 'UNIT'
        if 'CONDITION' in label or label.endswith('_COND'):
            return 'CONDITION'
        if any(kw in label for kw in ('PARAM', 'NAME', 'PIN', 'SYMBOL')):
            return 'PARAMETER'

        return 'VALUE'

    # ------------------------------------------------------------------
    # Tokenization
    # ------------------------------------------------------------------

    def _tokenize(self, text: str) -> List[Token]:
        if self.custom_tokenizer:
            return self._tokenize_custom(text)
        return self._tokenize_simple(text)

    def _tokenize_simple(self, text: str) -> List[Token]:
        """
        FIX #3: number and unit are always split into separate tokens.
        Pattern order:
          1. Structural  : \\t or \\n
          2. Sci-notation: digits with optional exponent (e/E + sign + digits) — NO trailing letters
          3. Plain number: integer or decimal — NO trailing letters
          4. Unit / word : letter-only run (catches µ, °, Ω, ±)
          5. Punctuation : everything else non-whitespace
        """
        tokens: List[Token] = []
        pattern = re.compile(
            r'(\t|\n)'                                          # 1. structural
            r'|([+-]?(?:\d+\.?\d*|\.\d+)[eE][+-]?\d+)'        # 2. sci-notation number
            r'|([+-]?(?:\d+\.?\d*|\.\d+))'                     # 3. plain number
            r'|([a-zA-ZÀ-ÿµ°Ω±%]+(?:[-/][a-zA-ZÀ-ÿ]+)*)'    # 4. word / unit
            r'|([^\w\s])'                                       # 5. punctuation
        )
        for m in pattern.finditer(text):
            tok_text = m.group(0)
            if tok_text.strip() == '' and tok_text not in ('\t', '\n'):
                continue
            tokens.append(Token(text=tok_text, start=m.start(), end=m.end()))
        return tokens

    def _tokenize_custom(self, text: str) -> List[Token]:
        tok = self.custom_tokenizer
        tokens: List[Token] = []

        if hasattr(tok, 'tokenize'):
            cursor = 0
            for rt in tok.tokenize(text):
                clean = rt.replace('##', '')
                pos = text.find(clean, cursor)
                if pos == -1:
                    continue
                tokens.append(Token(text=rt, start=pos, end=pos + len(clean)))
                cursor = pos + len(clean)
        elif callable(tok):
            cursor = 0
            for rt in tok(text):
                pos = text.find(rt, cursor)
                if pos == -1:
                    continue
                tokens.append(Token(text=rt, start=pos, end=pos + len(rt)))
                cursor = pos + len(rt)

        return tokens

    # ------------------------------------------------------------------
    # BIO tag assignment
    # ------------------------------------------------------------------

    def _assign_bio_tags(self):
        span_list = sorted(self.spans, key=lambda s: s.start)
        span_idx = 0
        n_spans = len(span_list)

        # Track which span the previous token belonged to, so we can
        # distinguish a true B- (first token in span) from an I- (continuation).
        current_span_id = None

        for token in self.tokens:
            while span_idx < n_spans and span_list[span_idx].end <= token.start:
                span_idx += 1

            if span_idx < n_spans:
                sp = span_list[span_idx]
                if token.start >= sp.start and token.end <= sp.end:
                    span_id = id(sp)

                    # FIX #2 (orphan I- bug): use span identity, not character equality.
                    # token.start == sp.start fails when the span starts with whitespace
                    # (e.g. <span data-label="MAX"> 75</span>) because the tokenizer
                    # skips the leading space, so token.start > sp.start even for the
                    # very first token.  Tracking span identity is reliable regardless.
                    if span_id != current_span_id:
                        token.span_label = f"B-{sp.label}"
                        current_span_id = span_id
                    else:
                        token.span_label = f"I-{sp.label}"
                    continue

            token.span_label = "O"
            current_span_id = None

    # ------------------------------------------------------------------
    # Sliding window (stride = max_tokens - overlap)
    # ------------------------------------------------------------------

    def _sliding_window(self, max_tokens: int = 512, overlap: int = 50) -> List[Dict]:
        if not self.tokens:
            return []

        stride = max_tokens - overlap
        samples = []
        n = len(self.tokens)
        start = 0

        while start < n:
            end = min(start + max_tokens, n)
            window = self.tokens[start:end]

            ner_tags = [t.span_label for t in window]

            # FIX #2 (window-boundary I- bug): if the window starts mid-entity
            # (e.g. "Power" with tag I-PARAMETER), promote the first token to B-
            # so every window begins with a valid BIO sequence.
            if ner_tags and ner_tags[0].startswith("I-"):
                ner_tags[0] = "B-" + ner_tags[0][2:]

            samples.append({
                'id': f'sample_{len(samples)}',
                'text': self.clean_text,
                'tokens': [t.text for t in window],
                'ner_tags': ner_tags,
                'token_start_idx': start,
                'token_end_idx': end - 1,
                'relations': [],
            })

            if end >= n:
                break
            start += stride

        return samples

    # ------------------------------------------------------------------
    # Relation extraction
    # ------------------------------------------------------------------

    def _add_relations(self, samples: List[Dict], jsonl_relations: List[Dict]) -> List[Dict]:
        if not jsonl_relations:
            return samples

        processed: List[Dict] = []
        for rel in jsonl_relations:
            subj = rel.get('subject', '')
            pred = rel.get('predicate', '')
            obj  = rel.get('object', '')

            if isinstance(subj, list):
                subj = ' '.join(str(x) for x in subj)
            if isinstance(obj, list):
                obj = ' '.join(str(x) for x in obj)

            subj, obj = str(subj).strip(), str(obj).strip()
            if subj and obj:
                processed.append({
                    'subject': subj,
                    'predicate': self._map_predicate(pred),
                    'object': obj,
                })

        if not processed:
            return samples

        def normalize(s: str) -> str:
            return re.sub(r'[^a-z0-9]', '', s.lower())

        # Build inverted index
        # FIX: index only tokens that belong to a real entity (non-O tag).
        # Without this guard, the same word (e.g. "Resistance") found in
        # free-text description paragraphs (tag=O) and in the table (tag=PARAMETER)
        # would both enter the index, and the proximity matcher could link a
        # relation to the untagged free-text occurrence instead of the entity.
        phrase_index: Dict[str, List[Tuple[int, List[int]]]] = {}

        for si, sample in enumerate(samples):
            toks      = sample['tokens']
            tags      = sample['ner_tags']
            max_ngram = min(10, len(toks))
            for n in range(1, max_ngram + 1):
                for i in range(len(toks) - n + 1):
                    # At least the first token of the n-gram must be an entity token
                    if tags[i] == 'O':
                        continue
                    phrase_norm = normalize(' '.join(toks[i:i + n]))
                    if not phrase_norm:
                        continue
                    phrase_index.setdefault(phrase_norm, []).append((si, list(range(i, i + n))))

        # Match relations
        for rel in processed:
            subj_norm = normalize(rel['subject'])
            obj_norm  = normalize(rel['object'])

            for sample_idx, subj_indices in phrase_index.get(subj_norm, []):
                obj_matches = [
                    (idx, idxs) for idx, idxs in phrase_index.get(obj_norm, [])
                    if idx == sample_idx
                ]
                if not obj_matches:
                    continue

                subj_end = subj_indices[-1]
                _, obj_indices = min(obj_matches, key=lambda x: abs(x[1][0] - subj_end))

                samples[sample_idx]['relations'].append({
                    'head': subj_indices,
                    'type': rel['predicate'],
                    'tail': obj_indices,
                })
                break

        return samples

    def _map_predicate(self, raw_pred: str) -> str:
        p = raw_pred.lower().strip()
        if 'unit' in p:
            return 'has_unit'
        if 'min' in p:
            return 'has_min'
        if 'max' in p:
            return 'has_max'
        if 'typ' in p:
            return 'has_typ'
        if 'cond' in p:
            return 'has_condition'
        return 'has_value'


# ---------------------------------------------------------------------------
# File-level helpers
# ---------------------------------------------------------------------------

def process_file(
    html_path: str,
    jsonl_path: str = None,
    output_path: str = None,
) -> List[Dict]:
    with open(html_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    relations: List[Dict] = []
    if jsonl_path and os.path.exists(jsonl_path):
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    relations.append(json.loads(line))

    engine = PreprocessingEngine()
    samples = engine.process(html_content, relations)

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(samples, f, indent=2, ensure_ascii=False)

    return samples


def run_batch_processing(
    input_html_dir: str,
    input_jsonl_dir: str,
    output_dataset_dir: str,
    debug_interval: int = 100,
):
    print(f"\n{'='*70}\nBATCH PROCESSING START\n{'='*70}")
    os.makedirs(output_dataset_dir, exist_ok=True)

    print("\n[PHASE 1] Loading JSONL relations...")
    relations_map: Dict[str, List[Dict]] = {}
    for jsonl_file in glob.glob(os.path.join(input_jsonl_dir, "*.jsonl")):
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                sid = data.get('sample_id')
                rels = data.get('relation_ground_truth', [])
                if sid:
                    relations_map[sid] = rels
    print(f"  ✓ {len(relations_map)} samples with relations loaded.")

    html_files = glob.glob(os.path.join(input_html_dir, "*.html"))
    print(f"\n[PHASE 2] Found {len(html_files)} HTML files.")
    print(f"\n[PHASE 3] Processing...")

    success = errors = 0
    phase_start = time.time()

    for idx, html_path in enumerate(html_files, 1):
        fname = os.path.basename(html_path)
        sample_id = fname.replace('datasheet_', '').replace('.html', '')
        sample_relations = relations_map.get(sample_id, [])

        if idx % debug_interval == 0 or idx == 1:
            elapsed = time.time() - phase_start
            avg = elapsed / idx
            eta = (len(html_files) - idx) * avg
            print(f"\n  [{idx}/{len(html_files)}] {idx/len(html_files)*100:.1f}% | "
                  f"elapsed={elapsed:.1f}s avg={avg:.3f}s/file ETA={eta:.1f}s "
                  f"ok={success} err={errors}")

        try:
            with open(html_path, 'r', encoding='utf-8') as f:
                html_content = f.read()

            engine = PreprocessingEngine()
            samples = engine.process(html_content, sample_relations)

            out_path = os.path.join(output_dataset_dir, f"{sample_id}.json")
            with open(out_path, 'w', encoding='utf-8') as f:
                json.dump(samples, f, indent=2, ensure_ascii=False)

            success += 1
        except Exception as exc:
            errors += 1
            print(f"  [ERROR] {fname}: {exc}")

    total = time.time() - phase_start
    print(f"\n{'='*70}")
    print(f"DONE — {success}/{len(html_files)} files OK | {errors} errors")
    print(f"Total: {total:.1f}s  avg: {total/max(len(html_files),1):.3f}s/file")
    print(f"Output: {output_dataset_dir}\n{'='*70}\n")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    INPUT_HTML_DIR  = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\html"
    INPUT_JSONL_DIR = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset"
    OUTPUT_DIR      = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\json_output"

    run_batch_processing(INPUT_HTML_DIR, INPUT_JSONL_DIR, OUTPUT_DIR, debug_interval=100)

Writing C:/Users/nivsa/Generation of Synthetic Training Data/embedded/extraction_engine/aligner.py


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 
     r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\extraction_engine\train_joint_model.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

הרצת הבדיקה

In [2]:
import os
import glob
import json
from collections import defaultdict

def validate_dataset(data_dir: str):
    print(f"🔍 Starting validation on directory: {data_dir}\n")
    
    json_files = glob.glob(os.path.join(data_dir, "*.json"))
    if not json_files:
        print(f"❌ No JSON files found in {data_dir}")
        return
    
    total_samples = 0
    errors = defaultdict(list)
    
    for filepath in json_files:
        filename = os.path.basename(filepath)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            # תמיכה גם בקובץ שהוא רשימה של דגימות וגם בדגימה בודדת
            if not isinstance(data, list):
                data = [data]
                
            for idx, sample in enumerate(data):
                total_samples += 1
                sample_id = sample.get("id", f"{filename}_sample_{idx}")
                
                # 1. בדיקה שכל המפתחות קיימים
                for key in ["tokens", "ner_tags", "relations"]:
                    if key not in sample:
                        errors["Missing Keys"].append(f"{sample_id}: missing '{key}'")
                
                if "Missing Keys" in errors and errors["Missing Keys"][-1].startswith(sample_id):
                    continue # דילוג על דגימה חסרה לחלוטין
                        
                tokens = sample["tokens"]
                ner_tags = sample["ner_tags"]
                relations = sample["relations"]
                
                # 2. חוק האורך: מספר הטוקנים חייב להיות זהה למספר התגיות
                if len(tokens) != len(ner_tags):
                    errors["Length Mismatch"].append(f"{sample_id}: {len(tokens)} tokens vs {len(ner_tags)} tags")
                    
                # 3. חוקי BIO: תגית I חייבת לבוא אחרי B או I מאותו סוג
                for i, tag in enumerate(ner_tags):
                    if tag.startswith("I-"):
                        base_tag = tag[2:]
                        if i == 0:
                            errors["Invalid BIO"].append(f"{sample_id}: Starts with I- tag at index 0 ('{tokens[i]}')")
                        else:
                            prev_tag = ner_tags[i-1]
                            if prev_tag == "O" or prev_tag[2:] != base_tag:
                                errors["Invalid BIO"].append(f"{sample_id}: I-{base_tag} follows {prev_tag} at index {i} ('{tokens[i]}')")
                                
                # 4. חוקי קשרים (Relations)
                for r_idx, rel in enumerate(relations):
                    head = rel.get("head", [])
                    tail = rel.get("tail", [])
                    
                    if not head or not tail:
                        errors["Empty Relation"].append(f"{sample_id}: Relation {r_idx} has empty head or tail")
                        continue
                        
                    # א. חריגה מגבולות המערך
                    if max(head) >= len(tokens) or max(tail) >= len(tokens):
                        errors["Relation Out of Bounds"].append(f"{sample_id}: Relation {r_idx} indices exceed token length")
                        continue
                        
                    # ב. בדיקה לוגית: האם ה-Head בכלל מצביע לישות? (הוא לא אמור להיות "O")
                    head_start_tag = ner_tags[head[0]]
                    if head_start_tag == "O":
                        errors["Relation to Non-Entity"].append(f"{sample_id}: Relation head points to 'O' at index {head[0]} ('{tokens[head[0]]}')")

        except Exception as e:
            errors["File Parsing Error"].append(f"{filename}: {str(e)}")

    # ---------------------------------------------------------
    # דוח סופי
    # ---------------------------------------------------------

    print("=" * 40)
    print("📊 VALIDATION REPORT")
    print("=" * 40)
    print(f"Files processed:   {len(json_files):,}")
    print(f"Samples checked:   {total_samples:,}")
    print("=" * 40 + "\n")
    
    if not errors:
        print("✅ PERFECT! 0 errors found.")
        print("Your dataset is 100% clean, consistent, and ready for production training! 🚀")
    else:
        print("🚨 ERRORS FOUND:")
        for err_type, err_list in errors.items():
            print(f"\n[{err_type}] - {len(err_list)} issues found:")
            # נדפיס רק את ה-5 הראשונים מכל סוג כדי לא להציף את המסך
            for e in err_list[:5]:
                print(f"  - {e}")
            if len(err_list) > 5:
                print(f"  ... and {len(err_list) - 5} more.")

if __name__ == '__main__':
    # זו הכתובת שכבר השתמשנו בה, תוודא רק שהיא עדיין מעודכנת אצלך:
    DATA_DIR = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\json_output"
    
    validate_dataset(DATA_DIR)

🔍 Starting validation on directory: C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\json_output

📊 VALIDATION REPORT
Files processed:   7,991
Samples checked:   11,862

✅ PERFECT! 0 errors found.
Your dataset is 100% clean, consistent, and ready for production training! 🚀


In [11]:
%%writefile "C:/Users/nivsa/Generation of Synthetic Training Data/embedded/extraction_engine/train_joint_model.py"

"""
Joint Entity & Relation Extraction Pipeline
Fine-tuning microsoft/deberta-v3-base on electronic datasheet NER + RE.
(Configured for Local CPU Sanity Check)

FIXES vs. previous version:
1. token_type_ids shape  — encoding.get() returned a dict key, not a tensor;
                           now extracted safely with a proper fallback.
2. collate_fn zero-rel   — when max_rels=0, stack() received empty lists and
                           crashed; guarded with an explicit max_rels==0 branch.
3. autocast CPU crash    — torch.cuda.amp.autocast raises on CPU in older
                           PyTorch versions; replaced with contextlib.nullcontext
                           when fp16=False.
"""

import os
import json
import glob
import random
import contextlib
from typing import List, Dict, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from transformers import DebertaV2Tokenizer, DebertaV2Model, get_linear_schedule_with_warmup
from torch.cuda.amp import GradScaler, autocast
import numpy as np

# ============================================================
# PART 1 — Configuration & Labels
# ============================================================

CONFIG = {
    "data_dir":       r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\sanity_check_json",
    "model_name":     "microsoft/deberta-v3-base",
    "output_dir":     r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\sanity_check_json\checkpoints_sanity",
    "max_length":     512,
    "batch_size":     2,
    "num_epochs":     1,
    "learning_rate":  2e-5,
    "weight_decay":   0.01,
    "warmup_ratio":   0.0,
    "alpha":          1.0,
    "beta":           0.5,
    "val_split":      0.2,
    "dropout":        0.1,
    "grad_clip":      1.0,
    "fp16":           False,    # must be False on CPU
    "seed":           42,
}

NER_LABELS = [
    "O",
    "B-PARAMETER", "I-PARAMETER",
    "B-VALUE",      "I-VALUE",
    "B-MIN",        "I-MIN",
    "B-MAX",        "I-MAX",
    "B-TYP",        "I-TYP",
    "B-UNIT",       "I-UNIT",
    "B-CONDITION",  "I-CONDITION",
]
NER2ID = {label: idx for idx, label in enumerate(NER_LABELS)}
ID2NER = {idx: label for label, idx in NER2ID.items()}

REL_LABELS = [
    "NO_RELATION",
    "has_value",
    "has_unit",
    "has_min",
    "has_max",
    "has_typ",
    "has_condition",
]
REL2ID = {label: idx for idx, label in enumerate(REL_LABELS)}
ID2REL = {idx: label for label, idx in REL2ID.items()}

IGNORE_INDEX = -100


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# PART 2 — Dataset & DataLoader
# ============================================================

def build_relation_targets(
    relations: List[Dict],
    word_to_first_subword: List[Optional[int]],
    word_to_last_subword:  List[Optional[int]],
    ner_tags: List[str],
    max_seq_len: int,
    neg_ratio: float = 1.0,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

    head_spans, tail_spans, rel_labels = [], [], []
    positive_pairs: set = set()

    for rel in relations:
        head_word_ids: List[int] = rel["head"]
        tail_word_ids: List[int] = rel["tail"]
        rel_type: str = rel["type"]

        h_start = word_to_first_subword[head_word_ids[0]]
        h_end   = word_to_last_subword[head_word_ids[-1]]
        t_start = word_to_first_subword[tail_word_ids[0]]
        t_end   = word_to_last_subword[tail_word_ids[-1]]

        if any(x is None for x in [h_start, h_end, t_start, t_end]):
            continue
        if max(h_end, t_end) >= max_seq_len:
            continue

        head_spans.append([h_start, h_end])
        tail_spans.append([t_start, t_end])
        rel_labels.append(REL2ID.get(rel_type, REL2ID["NO_RELATION"]))
        positive_pairs.add((head_word_ids[0], tail_word_ids[0]))
        positive_pairs.add((tail_word_ids[0], head_word_ids[0]))

    # Negative sampling
    entity_word_indices: List[int] = [
        i for i, tag in enumerate(ner_tags)
        if tag.startswith("B-")
        and word_to_first_subword[i] is not None
        and word_to_last_subword[i]  is not None
    ]
    num_negatives_target = max(1, int(len(head_spans) * neg_ratio))
    candidate_pairs = [
        (i, j) for i in entity_word_indices for j in entity_word_indices
        if i != j and (i, j) not in positive_pairs
    ]
    random.shuffle(candidate_pairs)

    for h_word, t_word in candidate_pairs[:num_negatives_target]:
        h_start = word_to_first_subword[h_word]
        h_end   = word_to_last_subword[h_word]
        t_start = word_to_first_subword[t_word]
        t_end   = word_to_last_subword[t_word]
        if max(h_end, t_end) >= max_seq_len:
            continue
        head_spans.append([h_start, h_end])
        tail_spans.append([t_start, t_end])
        rel_labels.append(REL2ID["NO_RELATION"])

    if not head_spans:
        return (
            torch.zeros((0, 2), dtype=torch.long),
            torch.zeros((0, 2), dtype=torch.long),
            torch.zeros((0,),   dtype=torch.long),
        )

    return (
        torch.tensor(head_spans,  dtype=torch.long),
        torch.tensor(tail_spans,  dtype=torch.long),
        torch.tensor(rel_labels,  dtype=torch.long),
    )


class DatasheetDataset(Dataset):
    def __init__(self, data_dir: str, tokenizer_name: str, max_length: int):
        self.max_length = max_length
        self.tokenizer  = DebertaV2Tokenizer.from_pretrained(tokenizer_name)
        self.samples    = self._load_all_samples(data_dir)

    def _load_all_samples(self, data_dir: str) -> List[Dict]:
        samples    = []
        json_files = glob.glob(os.path.join(data_dir, "*.json"))
        if not json_files:
            raise FileNotFoundError(f"No JSON files found in {data_dir}")
        for path in json_files:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list):
                samples.extend(data)
            else:
                samples.append(data)
        print(f"Loaded {len(samples)} samples from {len(json_files)} files.")
        return samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        sample        = self.samples[idx]
        words         = sample["tokens"]
        word_ner_tags = sample["ner_tags"]
        relations     = sample.get("relations", [])

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        word_ids = encoding.word_ids(batch_index=0)
        seq_len  = len(word_ids)

        # FIX #1: token_type_ids — DeBERTa-v3 does not return token_type_ids in
        # the encoding dict, so encoding.get("token_type_ids") returns None (not a
        # tensor), causing squeeze(0) to crash later.  Extract safely here.
        if "token_type_ids" in encoding and encoding["token_type_ids"] is not None:
            token_type_ids = encoding["token_type_ids"].squeeze(0)
        else:
            token_type_ids = torch.zeros(seq_len, dtype=torch.long)

        subword_labels        = []
        previous_word_id      = None
        word_to_first_subword = [None] * len(words)
        word_to_last_subword  = [None] * len(words)

        for subword_pos, word_id in enumerate(word_ids):
            if word_id is None:
                subword_labels.append(IGNORE_INDEX)
                continue

            if word_to_first_subword[word_id] is None:
                word_to_first_subword[word_id] = subword_pos
            word_to_last_subword[word_id] = subword_pos

            if word_id != previous_word_id:
                tag = word_ner_tags[word_id] if word_id < len(word_ner_tags) else "O"
                subword_labels.append(NER2ID.get(tag, NER2ID["O"]))
            else:
                subword_labels.append(IGNORE_INDEX)

            previous_word_id = word_id

        head_spans, tail_spans, rel_labels = build_relation_targets(
            relations, word_to_first_subword, word_to_last_subword, word_ner_tags, seq_len
        )

        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "token_type_ids": token_type_ids,
            "ner_labels":     torch.tensor(subword_labels, dtype=torch.long),
            "head_spans":     head_spans,
            "tail_spans":     tail_spans,
            "rel_labels":     rel_labels,
        }


def collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    input_ids      = torch.stack([b["input_ids"]      for b in batch])
    attention_mask = torch.stack([b["attention_mask"]  for b in batch])
    token_type_ids = torch.stack([b["token_type_ids"]  for b in batch])
    ner_labels     = torch.stack([b["ner_labels"]      for b in batch])

    max_rels = max((b["rel_labels"].size(0) for b in batch), default=0)

    # FIX #2: zero-relation batches — when every sample in the batch has no
    # relations, max_rels=0 and torch.stack() receives empty tensors that
    # produce a shape mismatch.  Return zero-sized tensors directly.
    if max_rels == 0:
        B = len(batch)
        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "ner_labels":     ner_labels,
            "head_spans":     torch.zeros(B, 0, 2, dtype=torch.long),
            "tail_spans":     torch.zeros(B, 0, 2, dtype=torch.long),
            "rel_labels":     torch.zeros(B, 0,    dtype=torch.long),
            "rel_mask":       torch.zeros(B, 0,    dtype=torch.bool),
        }

    batch_head, batch_tail, batch_rel, batch_mask = [], [], [], []
    for b in batch:
        n   = b["rel_labels"].size(0)
        pad = max_rels - n
        if n > 0:
            h = torch.cat([b["head_spans"], torch.zeros(pad, 2, dtype=torch.long)], dim=0)
            t = torch.cat([b["tail_spans"], torch.zeros(pad, 2, dtype=torch.long)], dim=0)
            r = torch.cat([b["rel_labels"], torch.zeros(pad,    dtype=torch.long)], dim=0)
        else:
            h = torch.zeros(max_rels, 2, dtype=torch.long)
            t = torch.zeros(max_rels, 2, dtype=torch.long)
            r = torch.zeros(max_rels,    dtype=torch.long)
        mask = torch.tensor([1] * n + [0] * pad, dtype=torch.bool)
        batch_head.append(h)
        batch_tail.append(t)
        batch_rel.append(r)
        batch_mask.append(mask)

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
        "ner_labels":     ner_labels,
        "head_spans":     torch.stack(batch_head),
        "tail_spans":     torch.stack(batch_tail),
        "rel_labels":     torch.stack(batch_rel),
        "rel_mask":       torch.stack(batch_mask),
    }


# ============================================================
# PART 3 — Model
# ============================================================

class JointNERRE(nn.Module):
    def __init__(self, model_name, num_ner_labels, num_rel_labels, dropout, alpha, beta):
        super().__init__()
        self.alpha, self.beta = alpha, beta
        self.encoder    = DebertaV2Model.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size
        self.ner_dropout = nn.Dropout(dropout)
        self.ner_head    = nn.Linear(hidden_size, num_ner_labels)
        self.re_dropout  = nn.Dropout(dropout)
        self.re_head     = nn.Linear(hidden_size * 2, num_rel_labels)

    @staticmethod
    def _mean_pool_span(hidden_states: torch.Tensor, spans: torch.Tensor) -> torch.Tensor:
        B, num_rels, _ = spans.shape
        H      = hidden_states.size(-1)
        pooled = torch.zeros(B, num_rels, H, device=hidden_states.device)
        for b in range(B):
            for r in range(num_rels):
                start = spans[b, r, 0].item()
                end   = spans[b, r, 1].item() + 1
                pooled[b, r] = hidden_states[b, start:end].mean(dim=0)
        return pooled

    def forward(
        self, input_ids, attention_mask, token_type_ids,
        ner_labels=None, head_spans=None, tail_spans=None,
        rel_labels=None, rel_mask=None,
    ):
        outputs         = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

        sequence_output = outputs.last_hidden_state.float()
        ner_logits = self.ner_head(self.ner_dropout(sequence_output))
        ner_loss   = (
            F.cross_entropy(ner_logits.view(-1, ner_logits.size(-1)), ner_labels.view(-1), ignore_index=IGNORE_INDEX)
            if ner_labels is not None else None
        )

        re_logits, re_loss = None, None
        if head_spans is not None and tail_spans is not None and head_spans.size(1) > 0:
            head_repr = self._mean_pool_span(sequence_output, head_spans)
            tail_repr = self._mean_pool_span(sequence_output, tail_spans)
            re_logits = self.re_head(self.re_dropout(torch.cat([head_repr, tail_repr], dim=-1)))

            if rel_labels is not None and rel_mask is not None:
                active_logits = re_logits[rel_mask]
                active_labels = rel_labels[rel_mask]
                re_loss = (
                    F.cross_entropy(active_logits, active_labels)
                    if active_labels.numel() > 0
                    else torch.tensor(0.0, device=input_ids.device, requires_grad=True)
                )

        total_loss = None
        if ner_loss is not None and re_loss is not None:
            total_loss = self.alpha * ner_loss + self.beta * re_loss
        elif ner_loss is not None:
            total_loss = ner_loss

        return {"loss": total_loss, "ner_loss": ner_loss, "re_loss": re_loss,
                "ner_logits": ner_logits, "re_logits": re_logits}


# ============================================================
# PART 4 — Training Loop
# ============================================================

def train_one_epoch(model, loader, optimizer, scheduler, scaler, device, config):
    model.train()
    total_loss = steps = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        # FIX #3: torch.cuda.amp.autocast() raises RuntimeError on CPU with older
        # PyTorch because it requires CUDA to be available.  Use nullcontext when
        # fp16=False so the forward pass runs in normal float32 precision.
        amp_ctx = torch.amp.autocast(device_type=device.type, enabled=config["fp16"])
        with amp_ctx:
            outputs = model(**batch)
            loss    = outputs["loss"]

        if config["fp16"]:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

        scheduler.step()
        total_loss += loss.item()
        steps      += 1
        print(f"\r  Step {steps}/{len(loader)} - Loss: {loss.item():.4f}", end="")

    print()
    return {"loss": total_loss / steps}


def main():
    set_seed(CONFIG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    os.makedirs(CONFIG["output_dir"], exist_ok=True)

    full_dataset = DatasheetDataset(CONFIG["data_dir"], CONFIG["model_name"], CONFIG["max_length"])
    val_size     = max(1, int(len(full_dataset) * CONFIG["val_split"]))
    train_size   = len(full_dataset) - val_size
    train_dataset, val_dataset = random_split(
        full_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(CONFIG["seed"]),
    )
    print(f"Sanity Check — Train: {train_size} | Val: {val_size}")
    
    val_indices = val_dataset.indices
    val_samples = [full_dataset.samples[i] for i in val_indices]

    
    val_save_path = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\model tests\val.json"
    
    with open(val_save_path, "w", encoding="utf-8") as f:
        json.dump(val_samples, f, ensure_ascii=False, indent=2)
    
    print(f"Val set saved: {len(val_samples)} samples → {val_save_path}")

    train_loader = DataLoader(
        train_dataset, batch_size=CONFIG["batch_size"],
        shuffle=True, collate_fn=collate_fn,
    )

    model = JointNERRE(
        CONFIG["model_name"], len(NER_LABELS), len(REL_LABELS),
        CONFIG["dropout"], CONFIG["alpha"], CONFIG["beta"],
    ).to(device)

    encoder_params = list(model.encoder.named_parameters())
    head_params    = list(model.ner_head.named_parameters()) + list(model.re_head.named_parameters())
    optimizer = AdamW([
        {"params": [p for _, p in encoder_params], "lr": CONFIG["learning_rate"]},
        {"params": [p for _, p in head_params],    "lr": CONFIG["learning_rate"] * 5},
    ], weight_decay=CONFIG["weight_decay"])

    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0,
        num_training_steps=len(train_loader) * CONFIG["num_epochs"],
    )
    scaler = torch.amp.GradScaler(device.type, enabled=CONFIG["fp16"]) if device.type == "cuda" else None

    print("\nStarting Sanity Check Training Loop...")
    for epoch in range(1, CONFIG["num_epochs"] + 1):
        print(f"\nEpoch {epoch}/{CONFIG['num_epochs']}")
        metrics = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, device, CONFIG)
        print(f"  [Train] Average Loss: {metrics['loss']:.4f}")

    print("\n✅ Sanity check complete! Architecture, data loading, and backward pass all work.")


if __name__ == "__main__":
    main()

Overwriting C:/Users/nivsa/Generation of Synthetic Training Data/embedded/extraction_engine/train_joint_model.py


In [12]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 
     r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\extraction_engine\train_joint_model.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

Using device: cpu
Loaded 35 samples from 22 files.
Sanity Check — Train: 28 | Val: 7

STDERR: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Traceback (most recent call last):
  File "C:\Users\nivsa\Generation of Synthetic Training Data\embedded\extraction_engine\train_joint_model.py", line 473, in <module>
    main()
  File "C:\Users\nivsa\Generation of Synthetic Training Data\embedded\extraction_engine\train_joint_model.py", line 438, in main
    print(f"Val set saved: {len(val_samples)} samples \u2192 {val_save_path}")
  File "C:\Users\nivsa\Anaconda3\envs\ollama-env\lib\encodings\cp1255.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 25: character maps to <undefined>



בדיקת התוצאות

In [ ]:
# ----------------------------------------------------------------------------
# Donut Inference for Jupyter Notebook
# ----------------------------------------------------------------------------

# --- Imports ---
import json
import os
import re
import torch
from pathlib import Path
from PIL import Image
from transformers import VisionEncoderDecoderModel, DonutProcessor
import matplotlib.pyplot as plt

# ============================================================================
# CONFIGURATION (Change these paths!)
# ============================================================================

# Path to your fine-tuned model folder (where train.py saved it)
MODEL_PATH = "./donut-datasheets-finetuned"

# Path to the image you want to test
IMAGE_PATH = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\images\datasheet_00eb2cd6.jpg"

# Device (cuda or cpu)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================================
# FUNCTIONS
# ============================================================================

def load_model(model_path, device="cuda"):
    print(f"Loading model from: {model_path}")
    try:
        processor = DonutProcessor.from_pretrained(model_path)
        model = VisionEncoderDecoderModel.from_pretrained(model_path)
    except OSError as e:
        print(f"[X] Error: Could not load model from {model_path}")
        print(f"  Details: {e}")
        return None, None

    model.to(device)
    model.eval()
    
    print(f"[OK] Model loaded on {device}")
    print(f"[OK] Input Resolution: {processor.image_processor.size}")
    return model, processor

def extract_json(image_path, model, processor, device="cuda"):
    print(f"\nProcessing: {os.path.basename(image_path)}")
    
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return {"error": str(e)}

    # Display image in notebook
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    # Prepare input
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
    task_prompt = "<s_datasheet>"
    decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids.to(device)

    # Generate
    print("Generating output...")
    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=1024,
            early_stopping=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            return_dict_in_generate=True,
        )

    # Decode
    seq = processor.tokenizer.batch_decode(outputs.sequences, skip_special_tokens=True)[0]
    clean_seq = seq.replace(task_prompt, "").strip()
    
    # Try parsing JSON
    try:
        return json.loads(clean_seq)
    except json.JSONDecodeError:
        # Fallback: Try to find JSON object in text
        match = re.search(r'\{.*\}', clean_seq, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except:
                pass
        
        # Fallback: Fix missing braces
        if clean_seq.startswith("{") and not clean_seq.endswith("}"):
            try:
                return json.loads(clean_seq + "}")
            except:
                pass
                
        return {"raw_output": clean_seq, "error": "Failed to parse JSON"}

# ============================================================================
# MAIN EXECUTION
# ============================================================================

# 1. Load Model (Only runs once if you keep the cell output)
if 'model' not in globals():
    model, processor = load_model(MODEL_PATH, DEVICE)

# 2. Run Inference
if model:
    result = extract_json(IMAGE_PATH, model, processor, DEVICE)
    
    print("\n" + "="*60)
    print("EXTRACTION RESULT")
    print("="*60)
    print(json.dumps(result, indent=2, ensure_ascii=False))

In [3]:
import json
import os

# שים פה את הנתיב לתיקייה שמכילה את כל קבצי ה-JSON שעברו Preprocessing
output_dir = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\json_output"

total_windows = 0
total_tokens = 0
total_relations = 0
total_entities = 0

for filename in os.listdir(output_dir):
    if filename.endswith(".json"):
        with open(os.path.join(output_dir, filename), 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # הקובץ מכיל רשימה של חלונות (Sliding Windows) למסמך
            for sample in data:
                total_windows += 1
                
                # ספירת טוקנים
                total_tokens += len(sample.get("tokens", []))
                
                # ספירת קשרים (Relations)
                total_relations += len(sample.get("relations", []))
                
                # ספירת ישויות: כל ישות מוגדרת על ידי תחילת תגית (B-)
                ner_tags = sample.get("ner_tags", [])
                entities_in_sample = sum(1 for tag in ner_tags if tag.startswith("B-"))
                total_entities += entities_in_sample

print("=== 📊 Dataset Statistics ===")
print(f"Total Documents / Sliding Windows: {total_windows:,}")
print(f"Avg Tokens per Window: {total_tokens / total_windows:.0f}" if total_windows > 0 else 0)
print(f"Total Annotated Entities: {total_entities:,}")
print(f"Total Semantic Relations: {total_relations:,}")

=== 📊 Dataset Statistics ===
Total Documents / Sliding Windows: 11,862
Avg Tokens per Window: 379
Total Annotated Entities: 518,824
Total Semantic Relations: 195,515


In [7]:
%%writefile "C:/Users/nivsa/Generation of Synthetic Training Data/embedded/extraction_engine/evaluate.py"
"""
evaluate.py — Evaluation script for JointNERRE model
------------------------------------------------------
Usage:
    python evaluate.py \
        --model_path "C:/Users/nivsa/Generation of Synthetic Training Data/embedded/final_dataset/model tests/best_model.pt" \
        --val_path   "C:/Users/nivsa/Generation of Synthetic Training Data/embedded/final_dataset/model tests/val.json" \
        --model_name "microsoft/deberta-v3-base" \
        --output_dir "./eval_results"
"""

import argparse
import json
import os
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import DebertaV2Model, DebertaV2TokenizerFast
from seqeval.metrics import classification_report as seq_classification_report
from sklearn.metrics import classification_report as sk_classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import seaborn as sns
import numpy as np

# ── constants (must match your training config) ──────────────────────────────
IGNORE_INDEX = -100

NER_LABEL2ID = {
    "O": 0,
    "B-PARAMETER": 1, "I-PARAMETER": 2,
    "B-VALUE":     3, "I-VALUE":     4,
    "B-TYP":       5, "I-TYP":       6,
    "B-UNIT":      7, "I-UNIT":      8,
    "B-CONDITION": 9, "I-CONDITION": 10,
    "B-MIN":      11, "I-MIN":       12,
    "B-MAX":      13, "I-MAX":       14,
}
NER_ID2LABEL = {v: k for k, v in NER_LABEL2ID.items()}

REL_LABEL2ID = {
    "NEGATIVE":       0,
    "has_value":      1,
    "has_unit":       2,
    "has_condition":  3,
    "has_min":        4,
    "has_max":        5,
    "has_typ":        6,
}
REL_ID2LABEL = {v: k for k, v in REL_LABEL2ID.items()}

MAX_TOKENS    = 512
MAX_REL_PAIRS = 64


# ── model (copy of your definition) ─────────────────────────────────────────
class JointNERRE(nn.Module):
    def __init__(self, model_name, num_ner_labels, num_rel_labels,
                 dropout=0.1, alpha=1.0, beta=0.5):
        super().__init__()
        self.alpha, self.beta = alpha, beta
        self.encoder     = DebertaV2Model.from_pretrained(model_name)
        hidden_size      = self.encoder.config.hidden_size
        self.ner_dropout = nn.Dropout(dropout)
        self.ner_head    = nn.Linear(hidden_size, num_ner_labels)
        self.re_dropout  = nn.Dropout(dropout)
        self.re_head     = nn.Linear(hidden_size * 2, num_rel_labels)

    @staticmethod
    def _mean_pool_span(hidden_states, spans):
        B, num_rels, _ = spans.shape
        H      = hidden_states.size(-1)
        pooled = torch.zeros(B, num_rels, H, device=hidden_states.device)
        for b in range(B):
            for r in range(num_rels):
                start = spans[b, r, 0].item()
                end   = spans[b, r, 1].item() + 1
                if end > start:
                    pooled[b, r] = hidden_states[b, start:end].mean(dim=0)
        return pooled

    def forward(self, input_ids, attention_mask, token_type_ids,
                ner_labels=None, head_spans=None, tail_spans=None,
                rel_labels=None, rel_mask=None):
        outputs         = self.encoder(input_ids=input_ids,
                                       attention_mask=attention_mask,
                                       token_type_ids=token_type_ids)
        sequence_output = outputs.last_hidden_state.float()
        ner_logits      = self.ner_head(self.ner_dropout(sequence_output))

        re_logits = None
        if head_spans is not None and tail_spans is not None and head_spans.size(1) > 0:
            head_repr = self._mean_pool_span(sequence_output, head_spans)
            tail_repr = self._mean_pool_span(sequence_output, tail_spans)
            re_logits = self.re_head(
                self.re_dropout(torch.cat([head_repr, tail_repr], dim=-1))
            )

        return {"ner_logits": ner_logits, "re_logits": re_logits}


# ── dataset ──────────────────────────────────────────────────────────────────
def span_to_start_end(token_indices):
    """Convert a list of token indices to (start, end) tuple."""
    return (min(token_indices), max(token_indices))


def build_rel_tensors(relations, seq_len, max_pairs=MAX_REL_PAIRS):
    """Build head_spans, tail_spans, rel_labels, rel_mask tensors from relations list."""
    head_spans  = torch.zeros(max_pairs, 2, dtype=torch.long)
    tail_spans  = torch.zeros(max_pairs, 2, dtype=torch.long)
    rel_labels  = torch.zeros(max_pairs, dtype=torch.long)
    rel_mask    = torch.zeros(max_pairs, dtype=torch.bool)

    for i, rel in enumerate(relations[:max_pairs]):
        h_start, h_end = span_to_start_end(rel["head"])
        t_start, t_end = span_to_start_end(rel["tail"])
        if h_end < seq_len and t_end < seq_len:
            head_spans[i] = torch.tensor([h_start, h_end])
            tail_spans[i] = torch.tensor([t_start, t_end])
            rel_labels[i] = REL_LABEL2ID.get(rel["type"], 0)
            rel_mask[i]   = True

    return head_spans, tail_spans, rel_labels, rel_mask


class DatasheetDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=MAX_TOKENS):
        self.samples   = data
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        tokens = sample["tokens"]
        tags   = sample["ner_tags"]

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        input_ids      = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        token_type_ids = encoding.get("token_type_ids",
                         torch.zeros_like(input_ids)).squeeze(0)
        word_ids       = encoding.word_ids(batch_index=0)

        # align NER labels — only first subword gets real label, rest → IGNORE
        ner_labels = torch.full((self.max_len,), IGNORE_INDEX, dtype=torch.long)
        prev_word  = None
        for i, wid in enumerate(word_ids):
            if wid is None:
                continue
            if wid != prev_word:
                label_str    = tags[wid] if wid < len(tags) else "O"
                ner_labels[i] = NER_LABEL2ID.get(label_str, 0)
            prev_word = wid

        # map original token indices → subword indices (first subword of each word)
        word_to_first_subword = {}
        prev_word = None
        for i, wid in enumerate(word_ids):
            if wid is not None and wid != prev_word:
                word_to_first_subword[wid] = i
            prev_word = wid

        # remap relation spans to subword space
        remapped_relations = []
        for rel in sample.get("relations", []):
            head_sw = [word_to_first_subword[t] for t in rel["head"]
                       if t in word_to_first_subword]
            tail_sw = [word_to_first_subword[t] for t in rel["tail"]
                       if t in word_to_first_subword]
            if head_sw and tail_sw:
                remapped_relations.append({
                    "head": head_sw,
                    "tail": tail_sw,
                    "type": rel["type"],
                })

        seq_len = attention_mask.sum().item()
        head_spans, tail_spans, rel_labels, rel_mask = build_rel_tensors(
            remapped_relations, seq_len
        )

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "ner_labels":     ner_labels,
            "head_spans":     head_spans,
            "tail_spans":     tail_spans,
            "rel_labels":     rel_labels,
            "rel_mask":       rel_mask,
        }


# ── evaluation helpers ───────────────────────────────────────────────────────
def decode_ner_predictions(all_preds, all_labels):
    """Convert flat token-level id lists → seqeval-compatible nested lists."""
    pred_seqs, true_seqs = [], []
    for preds, labels in zip(all_preds, all_labels):
        pred_seq, true_seq = [], []
        for p, l in zip(preds, labels):
            if l == IGNORE_INDEX:
                continue
            pred_seq.append(NER_ID2LABEL.get(p, "O"))
            true_seq.append(NER_ID2LABEL.get(l, "O"))
        pred_seqs.append(pred_seq)
        true_seqs.append(true_seq)
    return pred_seqs, true_seqs


def plot_confusion_matrix(cm, labels, title, save_path):
    fig, ax = plt.subplots(figsize=(max(6, len(labels)), max(5, len(labels) - 1)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved: {save_path}")


# ── main evaluation loop ─────────────────────────────────────────────────────
@torch.no_grad()
def run_evaluation(model, loader, device):
    model.eval()

    all_ner_preds,  all_ner_labels  = [], []
    all_rel_preds,  all_rel_labels  = [], []

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out   = model(
            input_ids      = batch["input_ids"],
            attention_mask = batch["attention_mask"],
            token_type_ids = batch["token_type_ids"],
            head_spans     = batch["head_spans"],
            tail_spans     = batch["tail_spans"],
        )

        # ── NER ──────────────────────────────────────────────────────────────
        ner_preds = out["ner_logits"].argmax(dim=-1).cpu().tolist()
        ner_labs  = batch["ner_labels"].cpu().tolist()
        all_ner_preds.extend(ner_preds)
        all_ner_labels.extend(ner_labs)

        # ── RE ───────────────────────────────────────────────────────────────
        if out["re_logits"] is not None:
            re_preds  = out["re_logits"].argmax(dim=-1)
            rel_mask  = batch["rel_mask"]
            rel_labs  = batch["rel_labels"]
            all_rel_preds.extend(re_preds[rel_mask].cpu().tolist())
            all_rel_labels.extend(rel_labs[rel_mask].cpu().tolist())

    return all_ner_preds, all_ner_labels, all_rel_preds, all_rel_labels


# ── main ─────────────────────────────────────────────────────────────────────
def main(args):
    os.makedirs(args.output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*60}")
    print(f"  Device : {device}")
    print(f"  Model  : {args.model_path}")
    print(f"  Data   : {args.val_path}")
    print(f"{'='*60}\n")

    # ── load data ────────────────────────────────────────────────────────────
    with open(args.val_path, encoding="utf-8") as f:
        raw = json.load(f)
    print(f"Loaded {len(raw)} validation samples.")

    tokenizer = DebertaV2TokenizerFast.from_pretrained(args.model_name)
    dataset   = DatasheetDataset(raw, tokenizer)
    loader    = DataLoader(dataset, batch_size=args.batch_size, shuffle=False)

    # ── load model ───────────────────────────────────────────────────────────
    model = JointNERRE(
        model_name     = args.model_name,
        num_ner_labels = len(NER_LABEL2ID),
        num_rel_labels = len(REL_LABEL2ID),
    ).to(device)

    state = torch.load(args.model_path, map_location=device)
    # handle both raw state_dict and checkpoint dicts
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model.load_state_dict(state)
    print("Model loaded successfully.\n")

    # ── run ──────────────────────────────────────────────────────────────────
    ner_preds, ner_labels, rel_preds, rel_labels = run_evaluation(model, loader, device)

    # ── NER report (seqeval — entity-level) ──────────────────────────────────
    pred_seqs, true_seqs = decode_ner_predictions(ner_preds, ner_labels)

    print("=" * 60)
    print("  NER — Entity-Level Report (seqeval)")
    print("=" * 60)
    ner_report = seq_classification_report(true_seqs, pred_seqs, digits=4)
    print(ner_report)

    ner_report_path = os.path.join(args.output_dir, "ner_report.txt")
    with open(ner_report_path, "w") as f:
        f.write(ner_report)
    print(f"  Saved: {ner_report_path}\n")

    # ── NER confusion matrix (token-level, ignoring IGNORE_INDEX) ────────────
    flat_true = [l for l in ner_labels if l != IGNORE_INDEX]
    flat_pred = [p for p, l in zip(ner_preds, ner_labels) if l != IGNORE_INDEX]
    used_ids  = sorted(set(flat_true + flat_pred))
    used_lbls = [NER_ID2LABEL[i] for i in used_ids]
    cm_ner    = confusion_matrix(flat_true, flat_pred, labels=used_ids)
    plot_confusion_matrix(
        cm_ner, used_lbls,
        "NER Confusion Matrix (token-level)",
        os.path.join(args.output_dir, "ner_confusion_matrix.png"),
    )

    # ── RE report ────────────────────────────────────────────────────────────
    if rel_preds:
        print("=" * 60)
        print("  Relation Extraction Report")
        print("=" * 60)
        rel_label_names = [REL_ID2LABEL[i]
                           for i in sorted(set(rel_labels + rel_preds))]
        re_report = sk_classification_report(
            rel_labels, rel_preds,
            target_names=rel_label_names,
            digits=4,
        )
        print(re_report)

        re_report_path = os.path.join(args.output_dir, "re_report.txt")
        with open(re_report_path, "w") as f:
            f.write(re_report)
        print(f"  Saved: {re_report_path}\n")

        # RE confusion matrix
        used_rel_ids  = sorted(set(rel_labels + rel_preds))
        used_rel_lbls = [REL_ID2LABEL[i] for i in used_rel_ids]
        cm_re         = confusion_matrix(rel_labels, rel_preds, labels=used_rel_ids)
        plot_confusion_matrix(
            cm_re, used_rel_lbls,
            "RE Confusion Matrix",
            os.path.join(args.output_dir, "re_confusion_matrix.png"),
        )
    else:
        print("No relation predictions found — skipping RE report.\n")

    # ── qualitative examples ─────────────────────────────────────────────────
    print("=" * 60)
    print("  Qualitative Examples (first 3 samples)")
    print("=" * 60)
    qual_path = os.path.join(args.output_dir, "qualitative_examples.txt")
    with open(qual_path, "w", encoding="utf-8") as f:
        for sample, pred_seq, true_seq in zip(raw[:3], pred_seqs[:3], true_seqs[:3]):
            f.write(f"\n{'─'*50}\n")
            f.write(f"ID: {sample['id']}\n")
            f.write(f"{'TOKEN':<25} {'PREDICTED':<18} {'GROUND TRUTH'}\n")
            f.write(f"{'─'*60}\n")
            tokens = [t for t in sample["tokens"] if t.strip()]
            for tok, pred, true in zip(tokens, pred_seq, true_seq):
                marker = "  ✗" if pred != true else ""
                f.write(f"{tok:<25} {pred:<18} {true}{marker}\n")
    print(f"  Saved: {qual_path}\n")

    print("=" * 60)
    print("  Evaluation complete. All results saved to:", args.output_dir)
    print("=" * 60)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_path", required=True,
                        help="Path to best_model.pt")
    parser.add_argument("--val_path",   required=True,
                        help="Path to validation JSON file")
    parser.add_argument("--model_name", default="microsoft/deberta-v3-base",
                        help="HuggingFace model name (must match training)")
    parser.add_argument("--output_dir", default="./eval_results",
                        help="Directory to save reports and plots")
    parser.add_argument("--batch_size", type=int, default=8)
    args = parser.parse_args()
    main(args)

Writing C:/Users/nivsa/Generation of Synthetic Training Data/embedded/extraction_engine/evaluate.py


In [10]:
import subprocess, sys

evaluate_script = r"C:/Users/nivsa/Generation of Synthetic Training Data/embedded/extraction_engine/evaluate.py"  

model_path = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\model tests\checkpoint_epoch_3.pt"  

val_path = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\model tests\val.json"  

output_dir = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\model tests\eval_results"  

result = subprocess.run(
    [sys.executable, evaluate_script,
     "--model_path", model_path,
     "--val_path",   val_path,
     "--output_dir", output_dir],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

KeyboardInterrupt: 

In [13]:
import os

base = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded"

for root, dirs, files in os.walk(base):
    # הדפסת רמת ההיררכיה
    level = root.replace(base, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

embedded/
  Donut_Datasheet_FineTuning.ipynb
  Gemini_Generated_Image_h0alkhh0alkhh0al.png
  Gemini_Generated_Image_kr0z7nkr0z7nkr0z.png
  pipline of the model.txt
  Preprocessing _Pipeline.png
  Project Overview Workflow and Methodology for Datasheet Extraction.pptx
  Synthetic_Datasheet_Pipeline.ipynb
  .ipynb_checkpoints/
    Donut_Datasheet_FineTuning-checkpoint.ipynb
    Synthetic_Datasheet_Pipeline-checkpoint.ipynb
  donut-datasheets-finetuned/
  extraction_engine/
    aligner.py
    evaluate.py
    train_joint_model.py
  final_dataset/
    json_output.zip
    production_20260222_110334.jsonl
    production_20260226_093915.jsonl
    html/
      datasheet_00052463.html
      datasheet_005ceeed.html
      datasheet_009a9303.html
      datasheet_00a21dbf.html
      datasheet_00ab8aa7.html
      datasheet_00b2dc67.html
      datasheet_00d97c0e.html
      datasheet_01020043.html
      datasheet_010a044b.html
      datasheet_0130553d.html
      datasheet_013b2aaf.html
      datasheet_0